In [1]:
import pandas as pd

X_train = pd.read_csv("../data/X_train.csv")

y_train = pd.read_csv(
    "../data/y_train.csv"
).squeeze()

In [2]:
import sys
import os

sys.path.append(
    os.path.abspath("..")
)

In [3]:
from sklearn.ensemble import RandomForestRegressor

from src.evaluate import rmse_cv

In [4]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf_rmse = rmse_cv(
    rf_model,
    X_train,
    y_train
).mean()

print(
    f"Random Forest RMSE: {rf_rmse:.5f}"
)

Random Forest RMSE: 0.13207


In [5]:
X_test = pd.read_csv(
    "../data/X_test.csv"
)

test = pd.read_csv(
    "../data/test.csv"
)

print(X_test.shape)
print(test.shape)

(1459, 301)
(1459, 80)


In [6]:
rf_model.fit(
    X_train,
    y_train
)

print("RF 訓練完成")

RF 訓練完成


In [7]:
from sklearn.model_selection import cross_val_predict

# 1. 計算 Random Forest 的 5-Fold OOF 預測值
print("正在計算 Random Forest OOF...")
oof_rf = cross_val_predict(
    rf_model, 
    X_train, 
    y_train, 
    cv=5, 
    n_jobs=-1
)

# 2. 存檔備用
pd.Series(oof_rf, name="SalePrice").to_csv("rf_oof_train.csv", index=False)
print("RF OOF 已儲存為 rf_oof_train.csv")

正在計算 Random Forest OOF...
RF OOF 已儲存為 rf_oof_train.csv


In [8]:
rf_pred_log = rf_model.predict(X_test)

print(rf_pred_log[:5])

[11.7285206  12.00172716 12.04017671 12.11115083 12.16449371]


In [9]:
import numpy as np

rf_pred = np.expm1(rf_pred_log)

print(rf_pred[:5])

[124059.00014354 163035.13776092 169425.87750639 181887.8077068
 191853.72198735]


In [10]:
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": rf_pred
})

submission.head()

,Id,SalePrice
0,1461,124059.000144
1,1462,163035.137761
2,1463,169425.877506
3,1464,181887.807707
4,1465,191853.721987


In [11]:
submission.to_csv(
    "../submissions/rf_submission.csv",
    index=False
)

print("RF Submission 已建立")

RF Submission 已建立
